# Knee single-coil T-sweep (U2 only)

Runs all four methods (DPS, ADPS, PiGDM, FA-KGD) on 192 knee single-coil slices
across R in {4, 8} and T in {10, 20, 50, 100}.

**Prereqs in `MyDrive/fastmri_artifacts/`:** `network-snapshot.pkl`, `knee_singlecoil_val.tar`

**Runtime -> T4 GPU.**


In [ ]:
# Cell 1 - verify GPU
!nvidia-smi | head -20


In [ ]:
# Cell 2 - clone repo + ADPS dnnlib/torch_utils (needed by EDM pickle).
import os
%cd /content
if os.path.isdir('/content/fastmri/.git'):
    %cd /content/fastmri
    !git fetch --quiet && git reset --hard origin/main
else:
    !rm -rf fastmri
    !git clone https://github.com/carlo-scr/fastmri.git
    %cd /content/fastmri
if not os.path.isdir('external/adps/dnnlib'):
    !rm -rf external/adps
    !git clone --depth 1 https://github.com/utcsilab/ambient-diffusion-mri.git external/adps
!git --no-pager log -1 --oneline


In [ ]:
# Cell 3 - install deps (do NOT pin numpy; Colab's scikit-image needs >=2.3).
!pip install -q h5py s3fs wandb pyyaml fastmri


In [ ]:
# Cell 4 - mount Drive, stage knee data + EDM checkpoint.
# Expects in MyDrive/fastmri_artifacts/:
#   - network-snapshot.pkl  (EDM checkpoint, 250 MB)
#   - knee_singlecoil_val.tar  (knee multicoil/singlecoil val tarball; rename to whatever you have)
from google.colab import drive
drive.mount('/content/drive')
import os, shutil
ART = '/content/drive/MyDrive/fastmri_artifacts'
assert os.path.exists(f'{ART}/network-snapshot.pkl'), 'EDM checkpoint missing in Drive'
KNEE_TAR = f'{ART}/knee_singlecoil_val.tar'
assert os.path.exists(KNEE_TAR), f'knee tarball missing: {KNEE_TAR}'

os.makedirs('checkpoints/edm/supervised_R=1', exist_ok=True)
shutil.copy(f'{ART}/network-snapshot.pkl', 'checkpoints/edm/supervised_R=1/network-snapshot.pkl')

# Tarball should preserve data/singlecoil_val/... paths.
!tar xf "$KNEE_TAR" -C /content/fastmri
!find data/singlecoil_val -name '._*' -delete 2>/dev/null
!find data/singlecoil_val -name '.DS_Store' -delete 2>/dev/null

import h5py, glob
files = sorted(glob.glob('data/singlecoil_val/*.h5'))
bad = []
for f in files:
    try:
        with h5py.File(f, 'r'): pass
    except Exception as e:
        bad.append((f, str(e)))
print(f'{len(files)} h5 files; {len(bad)} unreadable')
for f, e in bad: print('  BAD:', f, '->', e)
!ls -la checkpoints/edm/supervised_R=1/


In [ ]:
# Cell U2 - Knee single-coil T-sweep on 192 slices, all 4 methods.
import os, subprocess, time
REPO    = '/content/fastmri'
DATA    = 'data/singlecoil_val'
CKPT    = 'checkpoints/edm/supervised_R=1'
NSL     = 192
CF      = {4: 0.08, 8: 0.04}
TAG     = 'fd_b5_smax10_clamp_oracle_v4_knee'
OUTROOT = f'outputs/Tsweep_{TAG}'
os.makedirs(f'{REPO}/{OUTROOT}', exist_ok=True)
for R in (4, 8):
    for T in (10, 20, 50, 100):
        out = f'{OUTROOT}/R{R}_T{T}'
        if os.path.exists(f'{REPO}/{out}/results.json'):
            print('  skip (exists):', out); continue
        print(f'\n===== knee R={R} T={T} -> {out} =====')
        t0 = time.time()
        cmd = ['python','-u','scripts/reconstruct.py',
            '--mode','edm','--checkpoint_dir',CKPT,'--data_path',DATA,
            '--num_slices',str(NSL),'--whole_volume',
            '--acceleration',str(R),'--center_fraction',str(CF[R]),
            '--num_steps',str(T),'--schedule','edm','--sigma_max','10.0',
            '--noise_init','oracle','--noise_model','freq_dep','--beta_noise','5.0',
            '--m_step_mode','clamp','--m_step_start_frac','0.0','--gamma','0.0',
            '--beta_fpdc','1.0','--alpha_ema','0.95',
            '--target_resolution','320','320','--device','cuda',
            '--methods','dps','adps','pigdm','fakgd',
            '--output_dir',out]
        r = subprocess.run(cmd, cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        print(r.stdout[-4000:])
        if r.returncode != 0:
            raise RuntimeError(f'rc={r.returncode} R={R} T={T}')
        print(f'  done in {(time.time()-t0)/60:.1f} min')


In [ ]:
# Cell - back up knee results to Drive.
import os, tarfile
ART = '/content/drive/MyDrive/fastmri_artifacts'
TAG = 'fd_b5_smax10_clamp_oracle_v4_knee'
src = f'outputs/Tsweep_{TAG}'
out_tar = f'{ART}/sweep_results_{TAG}.tar'
with tarfile.open(out_tar, 'w') as tf:
    if os.path.exists(src):
        tf.add(src, arcname=os.path.basename(src)); print('  +', src)
    else:
        print('  -', src, '(missing)')
print('Saved:', out_tar)
